<a href="https://colab.research.google.com/github/Fauzi455/PembelajaranMesin_2026/blob/main/JS-09/TUGAS/Tugas%20Lab2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TUGAS LAB 2

In [4]:
import pandas as pd
import numpy as np
import time
import faiss
from annoy import AnnoyIndex
import hnswlib
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
df = pd.read_csv('/content/drive/MyDrive/songs_with_attributes_and_lyrics.csv')
print(f"Bentuk data awal: {df.shape}")

Bentuk data awal: (955320, 17)


In [9]:
# Menghapus baris yang memiliki nilai NaN pada kolom fitur yang akan digunakan
features = ['danceability', 'energy', 'loudness', 'speechiness',
            'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']
df = df.dropna(subset=features).reset_index(drop=True)
print(f"Bentuk data setelah dropna: {df.shape}")

X = df[features].values

# Standarisasi fitur
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = X_scaled.astype('float32')

k = 10
d = X_scaled.shape[1]

# Indeks lagu query acak
query_index = 0
query_vector = X_scaled[query_index].reshape(1, -1)

# PERBAIKAN 1: Mencegah KeyError dengan mengecek apakah kolom tersedia
if 'track_name' in df.columns and 'track_artist' in df.columns:
    print(f"\nMencari {k} lagu terdekat untuk lagu: {df['track_name'].iloc[query_index]} - {df['track_artist'].iloc[query_index]}")
else:
    print(f"\nMencari {k} data terdekat untuk vektor pada indeks ke-{query_index}")


# Exact Nearest Neighbors (Scikit-Learn)
print("\n--- Exact NN (Scikit-Learn) ---")
exact_nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')

start_time = time.time()
exact_nn.fit(X_scaled)
# PERBAIKAN 2: Menambahkan operator pengurangan (-)
build_time_exact = time.time() - start_time

start_time = time.time()
distances_exact, indices_exact = exact_nn.kneighbors(query_vector)
search_time_exact = time.time() - start_time

print(f"Waktu Build Index: {build_time_exact:.6f} detik")
print(f"Waktu Pencarian: {search_time_exact:.6f} detik")
print("Hasil Indeks:", indices_exact[0])


# ANNOY (Approximate Nearest Neighbors Oh Yeah)
print("\n--- ANNOY ---")
n_trees = 50
annoy_index = AnnoyIndex(d, 'euclidean')

start_time = time.time()
for i in range(X_scaled.shape[0]):
    annoy_index.add_item(i, X_scaled[i])
annoy_index.build(n_trees)
build_time_annoy = time.time() - start_time

start_time = time.time()
indices_annoy, distances_annoy = annoy_index.get_nns_by_vector(query_vector[0], k, include_distances=True)
search_time_annoy = time.time() - start_time

print(f"Waktu Build Index: {build_time_annoy:.6f} detik (dengan {n_trees} trees)")
print(f"Waktu Pencarian: {search_time_annoy:.6f} detik")
print("Hasil Indeks:", indices_annoy)


# FAISS (Facebook AI Similarity Search)
print("\n--- FAISS (IndexFlatL2) ---")
faiss_index = faiss.IndexFlatL2(d)

start_time = time.time()
faiss_index.add(X_scaled)
build_time_faiss = time.time() - start_time

start_time = time.time()
distances_faiss, indices_faiss = faiss_index.search(query_vector, k)
search_time_faiss = time.time() - start_time

print(f"Waktu Build Index: {build_time_faiss:.6f} detik")
print(f"Waktu Pencarian: {search_time_faiss:.6f} detik")
print("Hasil Indeks:", indices_faiss[0])


# FAISS dengan Inverted File Index (IVF) untuk pencarian Approximate yang lebih cepat
print("\n--- FAISS (IVFFlat) ---")
nlist = 100
quantizer = faiss.IndexFlatL2(d)
faiss_ivf_index = faiss.IndexIVFFlat(quantizer, d, nlist, faiss.METRIC_L2)

start_time = time.time()
faiss_ivf_index.train(X_scaled)
faiss_ivf_index.add(X_scaled)
build_time_faiss_ivf = time.time() - start_time

# Menentukan jumlah sel yang akan dikunjungi saat mencari (nprobe)
faiss_ivf_index.nprobe = 10

start_time = time.time()
distances_faiss_ivf, indices_faiss_ivf = faiss_ivf_index.search(query_vector, k)
search_time_faiss_ivf = time.time() - start_time

print(f"Waktu Build Index (Train + Add): {build_time_faiss_ivf:.6f} detik (nlist={nlist})")
print(f"Waktu Pencarian: {search_time_faiss_ivf:.6f} detik (nprobe={faiss_ivf_index.nprobe})")
print("Hasil Indeks:", indices_faiss_ivf[0])


# HNSWlib (Hierarchical Navigable Small World)
print("\n--- HNSWlib ---")
hnsw_index = hnswlib.Index(space='l2', dim=d) # Menggunakan jarak L2 (Euclidean squared)

# Mengatur parameter indeks
num_elements = X_scaled.shape[0]
ef_construction = 200
M = 16

hnsw_index.init_index(max_elements=num_elements, ef_construction=ef_construction, M=M)

start_time = time.time()
hnsw_index.add_items(X_scaled)
build_time_hnsw = time.time() - start_time

# ef: Mengatur trade-off antara waktu pencarian vs akurasi pencarian
hnsw_index.set_ef(50)

start_time = time.time()
indices_hnsw, distances_hnsw = hnsw_index.knn_query(query_vector, k=k)
search_time_hnsw = time.time() - start_time

print(f"Waktu Build Index: {build_time_hnsw:.6f} detik (ef_construction={ef_construction}, M={M})")
print(f"Waktu Pencarian: {search_time_hnsw:.6f} detik (ef={hnsw_index.ef})")
print("Hasil Indeks:", indices_hnsw[0])


# Ringkasan perbandingan
print("\n" + "="*50)
print("RINGKASAN PERBANDINGAN WAKTU")
print("="*50)
summary = pd.DataFrame({
    'Metode': ['Exact NN (sklearn)', 'ANNOY', 'FAISS (FlatL2)', 'FAISS (IVFFlat)', 'HNSWlib'],
    'Build Waktu (detik)': [build_time_exact, build_time_annoy, build_time_faiss, build_time_faiss_ivf, build_time_hnsw],
    'Search Waktu (detik)': [search_time_exact, search_time_annoy, search_time_faiss, search_time_faiss_ivf, search_time_hnsw]
})

print(summary.to_string(index=False))


# Memeriksa irisan hasil untuk melihat seberapa akurat metode approximate dibandingkan exact NN
exact_set = set(indices_exact[0])
print("\nAkurasi Pencarian :")
print(f"ANNOY: {len(exact_set.intersection(set(indices_annoy)))} / {k}")
print(f"FAISS (FlatL2): {len(exact_set.intersection(set(indices_faiss[0])))} / {k}")
print(f"FAISS (IVFFlat): {len(exact_set.intersection(set(indices_faiss_ivf[0])))} / {k}")
print(f"HNSWlib: {len(exact_set.intersection(set(indices_hnsw[0])))} / {k}")

Bentuk data setelah dropna: (955320, 17)

Mencari 10 data terdekat untuk vektor pada indeks ke-0

--- Exact NN (Scikit-Learn) ---
Waktu Build Index: 0.024893 detik
Waktu Pencarian: 0.044634 detik
Hasil Indeks: [     0 394553 764272 837727 749223  61511 625227 833164 676733 523698]

--- ANNOY ---
Waktu Build Index: 85.605937 detik (dengan 50 trees)
Waktu Pencarian: 0.000894 detik
Hasil Indeks: [0, 394553, 764272, 837727, 749223, 61511, 625227, 833164, 676733, 523698]

--- FAISS (IndexFlatL2 - Exact Search as Baseline for FAISS) ---
Waktu Build Index: 0.087762 detik
Waktu Pencarian: 0.024240 detik
Hasil Indeks: [     0 394553 764272 837727 749223  61511 625227 833164 676733 523698]

--- FAISS (IVFFlat - Approximate Search) ---
Waktu Build Index (Train + Add): 0.539230 detik (nlist=100)
Waktu Pencarian: 0.004133 detik (nprobe=10)
Hasil Indeks: [     0 394553 764272 837727 749223  61511 625227 833164 676733 523698]

--- HNSWlib ---
Waktu Build Index: 232.560885 detik (ef_construction=200, 